# External model AUROC comparison

This notebook loads only the selected model checkpoints from `final_model_selection/checkpoints`, evaluates them on the CTU data below, and produces one row per model. The two AUROC columns report the point estimate and its patient-bootstrap 95% confidence interval at 0 and 20 minutes before delivery.

Every execution writes to a new timestamped external-results directory. No prior metrics, predictions, baseline outputs, or archived result artifacts are loaded.

In [ ]:
from __future__ import annotations

import json
import re
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

import utils
from methods.marked_survival import discover_ph_threshold_columns
from methods.Inception_classifier import make_model
from time_to_event_evaluation import (
    END_LANDMARK_SECONDS, EVALUATION_HORIZON_SECONDS, MAX_TRACING_SECONDS,
    evaluate_landmark_suite,
)

REPO_ROOT = Path.cwd().resolve()
sys.path.append(str(REPO_ROOT))

FINAL_SELECTION_DIR = REPO_ROOT / 'final_model_selection'
# Only selected InceptionTime checkpoint artifacts are evaluated here.
CHECKPOINT_DIR = FINAL_SELECTION_DIR / 'checkpoints'
# Every notebook execution gets an isolated output directory, so no prior
# metrics or predictions can be reused accidentally.
EXTERNAL_RESULT_ROOT = REPO_ROOT / 'results_final_model_selection_external_ctu'
EXTERNAL_RESULT_DIR = EXTERNAL_RESULT_ROOT / f'run_{datetime.now():%Y%m%d_%H%M%S}'
EVALUATOR = REPO_ROOT / 'postprocessing' / 'postprocess_ctu_checkpoints.py'
TRACE_FILE = REPO_ROOT / 'data/MFM_CTU_tracings.sample_limit-all.sample_rate-025.horizon-0.max_gap_hours-18.window-full.min_length-30.tail_length-2880.missingness-none.h5'
TRAIN_LABELS = REPO_ROOT / 'data/MFM_CTU_labs_train.parquet'
TEST_LABELS = REPO_ROOT / 'data/MFM_CTU_labs_test_all.parquet'
TABLE_OUTPUT = EXTERNAL_RESULT_DIR / 'auroc_at_0_20.csv'
LATEX_TABLE_OUTPUT = EXTERNAL_RESULT_DIR / 'tables' / 'auroc_at_0_20.tex'

# These defaults match the external CTU job and the primary TTE endpoint.
SPLIT = 'Test'
TARGET = 'macro_pH'
METRIC = 'auroc_end_landmark'
LANDMARK_MINUTES = (0.0, 20.0)
assert FINAL_SELECTION_DIR.is_dir(), FINAL_SELECTION_DIR
assert CHECKPOINT_DIR.is_dir(), CHECKPOINT_DIR
assert EVALUATOR.is_file(), EVALUATOR
for input_path in (TEST_LABELS,):
    assert input_path.is_file(), input_path


## Evaluate the packaged checkpoints on CTU

This runs the same `postprocess_ctu_checkpoints.py` evaluator used by the external-validation job wrapper for TTE checkpoints. The normal last-hour classifier is excluded because it is not a time-to-event model and has no TTE training-config sidecar.

In [ ]:
all_checkpoints = sorted(
    path for path in CHECKPOINT_DIR.glob('*/*.pt')
    if not path.name.endswith(('_optimizer.pt', '_training_config.pt'))
)
# The pretrained last-hour classifier is a separate classification-only
# artifact. Older packages use results_end_of_tracing_trained rather than
# results_classifier_last_hour, so recognise both locations.
CLASSIFIER_CHECKPOINT_DIRS = {'results_classifier_last_hour', 'results_end_of_tracing_trained'}
checkpoints = [
    path for path in all_checkpoints
    if not CLASSIFIER_CHECKPOINT_DIRS.intersection(path.parts)
]
excluded_checkpoints = [path for path in all_checkpoints if path not in checkpoints]
if excluded_checkpoints:
    print('Excluding non-TTE classifier checkpoint(s):')
    for path in excluded_checkpoints:
        print(f'  {path.relative_to(FINAL_SELECTION_DIR)}')
if not checkpoints:
    print('No selected TTE .pt checkpoints found; skipping deep-checkpoint CTU evaluation.')
def checkpoint_config_path(checkpoint: Path) -> Path | None:
    # Keep this in sync with postprocess_ctu_checkpoints.py. The config is
    # required model metadata, and must be packaged with the selected model.
    candidates = (
        checkpoint.with_name(checkpoint.stem + '_training_config.json'),
        checkpoint.parent.parent / (checkpoint.stem + '_training_config.json'),
    )
    direct = next((path for path in candidates if path.is_file()), None)
    if direct is not None:
        return direct
    return None

checkpoint_configs = {checkpoint: checkpoint_config_path(checkpoint) for checkpoint in checkpoints}
missing_sidecars = [checkpoint for checkpoint, config_path in checkpoint_configs.items() if config_path is None]
if missing_sidecars:
    raise FileNotFoundError(
        'Selected checkpoints without usable *_training_config.json files: '
        + ', '.join(str(path) for path in missing_sidecars)
    )
EXTERNAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(EVALUATOR)]
for checkpoint in checkpoints:
    # Pass supported TTE checkpoints explicitly so the evaluator cannot
    # rediscover the excluded classifier under the archive root.
    command.extend(['--checkpoint-dir', str(checkpoint)])
command.extend([
    '--trace-file', str(TRACE_FILE),
    '--train-labels', str(TRAIN_LABELS),
    '--test-labels', str(TEST_LABELS),
    '--output-dir', str(EXTERNAL_RESULT_DIR),
    '--gpu', '0',
    '--bootstrap-samples', '100',
    '--bootstrap-confidence-level', '0.95',
    '--fail-on-error',
])
print(f'Evaluating {len(checkpoints)} selected checkpoint(s):')
for checkpoint in checkpoints:
    print(f'  {checkpoint.relative_to(FINAL_SELECTION_DIR)}')
print(f'Writing CTU outputs to {EXTERNAL_RESULT_DIR}')
if checkpoints:
    for input_path in (TRACE_FILE, TRAIN_LABELS):
        assert input_path.is_file(), input_path
    completed = subprocess.run(command, cwd=REPO_ROOT, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f'External checkpoint evaluation failed with exit code {completed.returncode}')


In [ ]:
# The last-hour classifier is evaluated through the shared landmark
# adapter below, separately from the TTE checkpoint evaluator.
classifier_checkpoints = [
    path for path in all_checkpoints
    if CLASSIFIER_CHECKPOINT_DIRS.intersection(path.parts)
]
if not classifier_checkpoints:
    print('No last-hour classifier checkpoint found; skipping classifier evaluation.')
else:
    classifier_output_dir = EXTERNAL_RESULT_DIR / 'classifier_last_hour'
    for input_path in (TRACE_FILE, TRAIN_LABELS):
        assert input_path.is_file(), input_path
    classifier_output_dir.mkdir(parents=True, exist_ok=True)
    classifier_feature_stats_metadata, _ = utils.load_hdf5_metadata(
        str(TRACE_FILE), str(TRAIN_LABELS), True, True,
        ['pH Cord', 'Base Excess Cord'], horizon=0, lab_order_delay=0,
    )
    classifier_feature_stats = utils.compute_hdf5_feature_stats(
        str(TRACE_FILE), classifier_feature_stats_metadata['PID'].to_numpy(),
        features=('toco', 'fecg'), missing_data_method='ffill',
    )
    classifier_ph_thresholds = np.asarray([7.05, 7.10, 7.15, 7.20], dtype=float)
    discovered_thresholds = np.asarray([
        threshold for threshold, _ in discover_ph_threshold_columns(
            pd.read_parquet(TRAIN_LABELS).columns
        )
    ], dtype=float)
    if not np.allclose(discovered_thresholds, classifier_ph_thresholds):
        raise ValueError(
            f'Unexpected CTU pH thresholds: {discovered_thresholds.tolist()}'
        )
    classifier_target_names = [f'pH < {threshold:g}' for threshold in classifier_ph_thresholds]
    classifier_target_indices = list(range(len(classifier_target_names)))
    classifier_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def load_last_hour_classifier(checkpoint_path):
        try:
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        except TypeError:
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
        state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
        if not isinstance(state_dict, dict) or 'fc.weight' not in state_dict:
            raise TypeError(
                f'Expected a classifier state_dict with fc.weight: {checkpoint_path}'
            )
        output_shape = int(state_dict['fc.weight'].shape[0])
        if output_shape < len(classifier_target_names):
            raise ValueError(
                f'{checkpoint_path} has {output_shape} outputs; four pH outputs are required.'
            )
        model = make_model((900, 2), output_shape=output_shape)
        model.load_state_dict(state_dict, strict=True)
        return model.to(classifier_device).eval()

    class ClassificationOnlyLandmarkEstimator:
        def __init__(self, model, feature_stats, batch_size=1024):
            self.model = model
            self.feature_stats = feature_stats
            self.batch_size = int(batch_size)
            self.time_grid = np.unique(np.concatenate((
                END_LANDMARK_SECONDS,
                EVALUATION_HORIZON_SECONDS,
                [MAX_TRACING_SECONDS],
            )))

        @staticmethod
        def time_to_event_seconds(chunks):
            return np.full(len(chunks), np.inf, dtype=float)

        def __call__(self, chunks):
            chunks = np.asarray(chunks, dtype=np.float32)
            if chunks.ndim != 3 or chunks.shape[1:] != (900, 2):
                raise ValueError(f'Expected [N, 900, 2] chunks, got {chunks.shape}')
            scaled = np.stack([
                utils._apply_feature_stats(chunk, self.feature_stats) for chunk in chunks
            ]).astype(np.float32)
            probabilities = []
            with torch.no_grad():
                for start in range(0, len(scaled), self.batch_size):
                    batch = torch.from_numpy(
                        scaled[start:start + self.batch_size]
                    ).transpose(1, 2).to(classifier_device)
                    probabilities.append(
                        torch.sigmoid(self.model(batch)).cpu().numpy()[:, :4]
                    )
            probabilities = np.concatenate(probabilities, axis=0)
            result = {'time_grid': self.time_grid.copy()}
            for index, name in enumerate(classifier_target_names):
                result[name] = np.repeat(
                    probabilities[:, index, None], len(self.time_grid), axis=1
                )
            result['delivery'] = np.zeros((len(chunks), len(self.time_grid)), dtype=float)
            return result

    for classifier_checkpoint in classifier_checkpoints:
        result_stem = classifier_output_dir / f'{classifier_checkpoint.stem}_classifier_external'
        metrics_path = result_stem.with_name(result_stem.name + '_landmark_metrics.csv')
        predictions_path = result_stem.with_name(result_stem.name + '_landmark_predictions.csv')
        manifest_path = result_stem.with_suffix('.json')
        classifier_model = load_last_hour_classifier(classifier_checkpoint)
        classifier_estimator = ClassificationOnlyLandmarkEstimator(
            classifier_model, classifier_feature_stats
        )
        classifier_metrics, classifier_predictions = evaluate_landmark_suite(
            trace_file=str(TRACE_FILE),
            train_label_file=str(TRAIN_LABELS),
            test_label_file=str(TEST_LABELS),
            features=('toco', 'fecg'),
            chunk_window_size=900,
            lab_order_delay=0,
            prediction=classifier_estimator,
            target_names=classifier_target_names,
            target_indices=classifier_target_indices,
            include_delivery=True,
            splits_file=None,
            missing_data_method='ffill',
            chunk_missingness_max_fraction=0.30,
            bootstrap_samples=100,
            bootstrap_random_state=42,
        )
        classifier_metrics.to_csv(metrics_path, index=False)
        classifier_predictions.to_csv(predictions_path, index=False)
        manifest_path.write_text(json.dumps({
            'checkpoint': str(classifier_checkpoint),
            'evaluation': 'external_ctu_classification_only_landmark_adapter',
            'trace_file': str(TRACE_FILE),
            'train_labels': str(TRAIN_LABELS),
            'test_labels': str(TEST_LABELS),
            'features': ['toco', 'fecg'],
            'chunk_window_size': 900,
            'lab_order_delay': 0,
            'missing_data_method': 'ffill',
            'chunk_missingness_max_fraction': 0.30,
            'landmark_semantics': 'flat classifier probability repeated across finite evaluator cuts',
            'metrics_path': str(metrics_path),
            'predictions_path': str(predictions_path),
        }, indent=2) + '\n')
        print(f'Wrote classifier CTU outputs: {metrics_path}')


## Load and label metric artifacts

External checkpoint evaluation writes one `*_landmark_metrics.csv` per checkpoint. The loader keeps the checkpoint/configuration identity in the model label, so separate selected configurations remain separate rows. If repeated artifacts have the same label, their estimates are averaged and the corresponding reported CI endpoints are averaged as well.

In [ ]:
def artifact_stem(path: Path) -> str:
    suffix = '_landmark_metrics.csv'
    if not path.name.endswith(suffix):
        raise ValueError(f'Not a landmark-metrics file: {path}')
    stem = path.name[:-len(suffix)]
    # postprocess_ctu_checkpoints appends a path-derived identity.
    return re.sub(r'_ctu_external_[0-9a-f]+$', '', stem)


def model_label(path: Path) -> str:
    stem = artifact_stem(path)
    lower = stem.lower()
    family = 'Model'
    if 'ctg_fri_cox' in lower:
        family = 'Rule-based CTG-only FRI'
    elif 'xgboost_ctg' in lower:
        family = 'XGBoost FIGO CTG Cox'
    elif 'classifier' in lower and 'deephit' not in lower:
        family = 'InceptionTime classifier (last hour)'
    elif 'marked_corn_deephit' in lower:
        family = 'Marked DeepHit (CORN)'
    elif 'marked_deephit' in lower:
        family = 'Marked DeepHit'
    elif 'competing_deephit' in lower:
        family = 'Competing-risk DeepHit'
    elif 'survival_deephit' in lower:
        family = 'Survival DeepHit'
    short_id = stem.rsplit('_', 1)[-1]
    if len(short_id) < 6:
        short_id = ''
    detail = stem.replace('marked_deephit_', '').replace('competing_deephit_', '')
    detail = detail.replace('survival_deephit_', '').replace('marked_corn_deephit_', '')
    detail = re.sub(r'_[0-9a-f]{10,}$', '', detail)
    return f'{family} — {detail} ({short_id})' if detail else family


def load_metric_files(root: Path) -> pd.DataFrame:
    if not root.is_dir():
        print(f'No freshly generated external-results directory found: {root}')
        return pd.DataFrame()
    frames = []
    for path in sorted(root.glob('*/*_landmark_metrics.csv')):
        try:
            frame = pd.read_csv(path)
        except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
            print(f'Skipping {path}: {error}')
            continue
        if frame.empty:
            continue
        frame = frame.copy()
        frame['model'] = model_label(path)
        frame['source_kind'] = 'fresh_external_evaluation'
        frame['source_file'] = str(path)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


external_metrics = load_metric_files(EXTERNAL_RESULT_DIR)
metrics = external_metrics
print(f'Fresh external metric files loaded: {external_metrics.source_file.nunique() if not external_metrics.empty else 0}')
if metrics.empty:
    raise FileNotFoundError(
        f'No metric files found. Run the external validation command first: {EXTERNAL_RESULT_DIR}'
    )


In [ ]:
def format_ci(mean: float, lower: float, upper: float) -> str:
    values = (mean, lower, upper)
    if not all(np.isfinite(value) for value in values):
        return 'NA'
    return f'{mean:.3f} ({lower:.3f}–{upper:.3f})'


def build_comparison_table(frame: pd.DataFrame) -> pd.DataFrame:
    required = {'split', 'target', 'metric', 'minutes_before_delivery', 'value', 'ci_lower', 'ci_upper', 'model'}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f'Metric files are missing columns: {sorted(missing)}')
    selected = frame[
        frame['split'].eq(SPLIT)
        & frame['target'].eq(TARGET)
        & frame['metric'].eq(METRIC)
        & frame['minutes_before_delivery'].astype(float).isin(LANDMARK_MINUTES)
    ].copy()
    for column in ('minutes_before_delivery', 'value', 'ci_lower', 'ci_upper'):
        selected[column] = pd.to_numeric(selected[column], errors='coerce')
    selected = selected[np.isfinite(selected['value'])]
    if selected.empty:
        raise ValueError(
            f'No finite {METRIC} rows for split={SPLIT!r}, target={TARGET!r}, '
            f'landmarks={LANDMARK_MINUTES}'
        )
    rows = []
    for model, group in selected.groupby('model', sort=True):
        row = {'model': model}
        for minutes in LANDMARK_MINUTES:
            values = group[group['minutes_before_delivery'].eq(minutes)]
            if values.empty:
                row[f'auroc at {int(minutes)}'] = 'NA'
                continue
            row[f'auroc at {int(minutes)}'] = format_ci(
                values['value'].mean(),
                values['ci_lower'].mean(),
                values['ci_upper'].mean(),
            )
        rows.append(row)
    return pd.DataFrame(rows, columns=['model', 'auroc at 0', 'auroc at 20'])

comparison_table = build_comparison_table(metrics)
TABLE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
comparison_table.to_csv(TABLE_OUTPUT, index=False)

def _latex_escape(value):
    return (
        str(value)
        .replace('\\', r'\textbackslash{}')
        .replace('_', r'\_')
        .replace('&', r'\&')
        .replace('%', r'\%')
        .replace('#', r'\#')
    )


def write_latex_comparison_table(table):
    lines = [
        r'\begin{table}[t]',
        r'\centering',
        r'\caption{External CTU-cohort AUROC at 0 and 20 minutes before delivery.',
        r'Values are point estimates with patient-bootstrap 95\% confidence intervals.}',
        r'\label{tab:external-auroc-at-0-20}',
        r'\small',
        r'\begin{tabular}{lcc}',
        r'\toprule',
        r'Model & AUROC at 0 min & AUROC at 20 min \\',
        r'\midrule',
    ]
    for row in table.itertuples(index=False):
        cells = [
            _latex_escape(row.model),
            _latex_escape(row[1]).replace('–', '--'),
            _latex_escape(row[2]).replace('–', '--'),
        ]
        lines.append(' & '.join(cells) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}', ''])
    LATEX_TABLE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    LATEX_TABLE_OUTPUT.write_text('\n'.join(lines))
    print(f'Wrote {LATEX_TABLE_OUTPUT}')


write_latex_comparison_table(comparison_table)
display(comparison_table)
print(f'Wrote {TABLE_OUTPUT}')


The table above is built exclusively from the selected checkpoints evaluated on the CTU cohort in this notebook execution. It is saved as CSV alongside a paper-ready LaTeX source under the current timestamped `results_final_model_selection_external_ctu/run_*/tables/` directory.